In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

df = pd.read_csv("brfss_clean_2020_2024.csv")

race_map = {
    1: "NH-White", 2: "NH-Black", 3: "AIAN",
    4: "Asian", 5: "NHOPI", 6: "Other/Multiracial",
    7: "Hispanic"
}
age_map = {
    1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39",
    5: "40-44", 6: "45-49", 7: "50-54", 8: "55-59",
    9: "60-64", 10: "65-69", 11: "70-74", 12: "75-79", 13: "80+"
}
sex_map = {1: "Male", 2: "Female"}
education_map = {
    1: "Did not graduate high school",
    2: "Graduated high school",
    3: "Attended college or technical school",
    4: "Graduated college or technical school"
}
income_map = {
    1: "<15k", 2: "15k-25k", 3: "25k-35k",
    4: "35k-50k", 5: "50k-100k", 6: "100k-200k", 7: "200k+"
}

df["race_group"]   = df["_RACEPRV"].map(race_map)
df["age_group"]    = df["_AGEG5YR"].map(age_map)
df["sex"]          = df["_SEX"].map(sex_map)
df["education"]    = df["_EDUCAG"].map(education_map)
df["income_group"] = df["_INCOMG1"].map(income_map)

# Drop 2020 rows — income is NaN, can't be used in model
df_model = df.dropna(subset=["race_group", "age_group", "sex",
                               "education", "income_group", "obese"]).copy()

print("Full dataset:", len(df))
print("Model dataset (2021-2024):", len(df_model))
print("Dropped (2020):", len(df) - len(df_model))
print("States:", df_model["_STATE"].nunique())

Full dataset: 1622499
Model dataset (2021-2024): 1322240
Dropped (2020): 300259
States: 54


In [2]:
# Encode state as numeric code
le_state = LabelEncoder()
df_model["state_code"] = le_state.fit_transform(df_model["_STATE"])

# One-hot encode demographics
feature_cols_base = ["age_group", "sex", "education", "income_group", "race_group"]
df_encoded = pd.get_dummies(df_model[feature_cols_base], drop_first=True)

# Add state dummies as fixed effects
state_dummies = pd.get_dummies(df_model["state_code"], prefix="state", drop_first=True)
X = pd.concat([df_encoded, state_dummies], axis=1)
y = df_model["obese"].values
w = df_model["_LLCPWT_adjusted"].values  # use adjusted weights

feature_cols = [c for c in X.columns]

print("Feature matrix shape:", X.shape)
print("Fitting weighted logistic regression...")

model = LogisticRegression(max_iter=1000, solver="lbfgs", n_jobs=-1)
model.fit(X, y, sample_weight=w)

y_pred = model.predict_proba(X)[:, 1]
auc = roc_auc_score(y, y_pred, sample_weight=w)
print(f"\nWeighted Training AUC: {auc:.4f}")
print("Done.")

Feature matrix shape: (1322240, 81)
Fitting weighted logistic regression...


/Users/adityaailsingnani/Desktop/College/Capstone/BRFSS_final/brfss_env/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



Weighted Training AUC: 0.6336
Done.


In [3]:
# Load group summary
df_summary = pd.read_csv("brfss_group_summary.csv")
group_cols = ["age_group", "sex", "education", "income_group", "race_group"]

# Encode cells the same way as training data
cells = df_summary[group_cols].copy()
cells_encoded = pd.get_dummies(cells[group_cols], drop_first=True)

# Add any missing columns from training
for col in df_encoded.columns:
    if col not in cells_encoded.columns:
        cells_encoded[col] = 0
cells_encoded = cells_encoded[df_encoded.columns]

# Add state dummies — set all to 0 for national baseline
for col in state_dummies.columns:
    cells_encoded[col] = 0

# Predict
cells["obesity_rate_modeled"] = model.predict_proba(cells_encoded)[:, 1]

# Merge back
df_summary_modeled = df_summary.merge(cells, on=group_cols)

print("Shape:", df_summary_modeled.shape)
print("\nObesity rate comparison:")
print(f"  Raw      — mean: {df_summary_modeled['obesity_rate'].mean():.4f}, std: {df_summary_modeled['obesity_rate'].std():.4f}")
print(f"  Smoothed — mean: {df_summary_modeled['obesity_rate_smoothed'].mean():.4f}, std: {df_summary_modeled['obesity_rate_smoothed'].std():.4f}")
print(f"  Modeled  — mean: {df_summary_modeled['obesity_rate_modeled'].mean():.4f}, std: {df_summary_modeled['obesity_rate_modeled'].std():.4f}")
print(f"\n  Modeled range: {df_summary_modeled['obesity_rate_modeled'].min():.4f} to {df_summary_modeled['obesity_rate_modeled'].max():.4f}")
print(f"  Any NaN: {df_summary_modeled['obesity_rate_modeled'].isna().sum()}")

df_summary_modeled.to_csv("brfss_group_summary_modeled.csv", index=False)
print("\nsaved.")

Shape: (4953, 10)

Obesity rate comparison:
  Raw      — mean: 0.3396, std: 0.2262
  Smoothed — mean: 0.3439, std: 0.0962
  Modeled  — mean: 0.3500, std: 0.1172

  Modeled range: 0.0431 to 0.5791
  Any NaN: 0

saved.
